In [ ]:
!pip install krippendorff datasets pandas huggingface_hub --break-system-packages

In [2]:
"""
Inter-judge agreement (Krippendorff's alpha) + judge-score aggregation
for the Prompt B LLM-as-judge pipeline outputs.

Install deps:
    pip install krippendorff datasets pandas huggingface_hub --break-system-packages

Expected input shape (one row per item_id x judge_model):
    item_id, system, judge_model, timestamp, status, raw_response,
    parsed_json, error, recomputed_total_rules_count,
    recomputed_hallucination_rate, recomputed_hallucination_score,
    recomputed_contradiction_count, recomputed_atomicity_ratio,
    recomputed_atomicity_score, recomputed_testability_ratio,
    recomputed_testability_score, recomputed_jargon_free_ratio,
    recomputed_jargon_free_score, validation_issues, is_consistent

"""

import numpy as np
import os
import getpass
import pandas as pd
import krippendorff
from datasets import load_dataset

# --------------------------------------------------------------------------
# 1. CONFIG -- edit these for your setup
# --------------------------------------------------------------------------
HF_DATASETS = {
    "exp10": "businessrules/exp10_promptB_results",
    "gpt4.1": "businessrules/gpt4.1_promptB_results",
}
HF_SPLIT = "train"


HF_OUTPUT_REPOS = {
    "exp10": "businessrules/exp10_promptB_results_aggregated",
    "gpt4.1": "businessrules/gpt4.1_promptB_results_aggregated",
}
PUSH_TO_HUB = True   
PUSH_PRIVATE = True  

# Metrics to evaluate, and the correct Krippendorff level of measurement
# for each. This matters: alpha is computed differently depending on
# whether the scale is nominal, ordinal, interval, or ratio.
#   - *_score columns are ordinal (1-5 Likert-style buckets)
#   - *_count columns are bounded counts -> treat as interval (or ordinal
#     if you want to be conservative)
#   - *_rate / *_ratio columns are continuous 0-1 proportions -> ratio
METRIC_LEVELS = {
    "recomputed_hallucination_score": "ordinal",
    "recomputed_atomicity_score": "ordinal",
    "recomputed_testability_score": "ordinal",
    "recomputed_jargon_free_score": "ordinal",
    "recomputed_hallucination_rate": "ratio",
    "recomputed_atomicity_ratio": "ratio",
    "recomputed_testability_ratio": "ratio",
    "recomputed_jargon_free_ratio": "ratio",
    "recomputed_contradiction_count": "interval",
    "recomputed_total_rules_count": "interval",
}

ITEM_COL = "item_id"
JUDGE_COL = "judge_model"
STATUS_COL = "status"  


# --------------------------------------------------------------------------
# 2. AUTHENTICATION -- Prompt B repos require a Hugging Face token
# --------------------------------------------------------------------------

def get_hf_token() -> str:
    """
    Resolve a Hugging Face access token.

    Checks the HF_TOKEN environment variable first (useful for CI /
    non-interactive runs); otherwise prompts interactively with getpass
    so the token is never echoed or persisted in the notebook. Unlike
    Prompt A's datasets, Prompt B's repos are gated/private, so a token
    is mandatory here -- the notebook will refuse to continue without one.
    """
    token = "hf_token" # removed for safety reasons
    if token:
        return token

    token = getpass.getpass(
        "Hugging Face access token required for Prompt B datasets "
        "(create one at https://huggingface.co/settings/tokens): "
    )
    if not token:
        raise ValueError(
            "A Hugging Face access token is required to load the Prompt B "
            "datasets. Set the HF_TOKEN environment variable or re-run "
            "this cell and paste a valid token when prompted."
        )
    return token


# --------------------------------------------------------------------------
# 3. LOADING
# --------------------------------------------------------------------------

def _null_columns_to_string(table: "pa.Table") -> "pa.Table":
    """
    Cast any Arrow "null"-typed columns to string.

    A column that is entirely None in one parquet shard gets inferred by
    PyArrow as type `null`. If another shard has real string values in
    that same column (e.g. `error`, which is empty for every successful
    judging call but populated for failures), concatenating/casting the
    two together raises `TypeError: Couldn't cast array of type string to
    null`. This is exactly the `datasets`-library error you hit. Casting
    null -> string up front avoids it entirely.
    """
    import pyarrow as pa

    new_cols = []
    for col, field in zip(table.columns, table.schema):
        if pa.types.is_null(field.type):
            col = col.cast(pa.string())
        new_cols.append(col)
    return pa.Table.from_arrays(new_cols, names=table.schema.names)


def load_judged_dataset(repo_id: str, token: str, split: str = HF_SPLIT) -> pd.DataFrame:
    """
    Load a judged dataset from the Hugging Face Hub.

    This reads the split's parquet shard(s) directly with
    `huggingface_hub` + `pyarrow` instead of going through
    `datasets.load_dataset()`. That sidesteps a `datasets`-library bug
    where a column that's entirely null in one shard (commonly `error`,
    since it's empty for every row where judging succeeded) fails to
    reconcile against another shard where the same column holds real
    strings -- exactly the `DatasetGenerationError` /
    `TypeError: Couldn't cast array of type string to null` you saw.

    `token` is required (Prompt B repos are gated/private) and is passed
    explicitly to every Hub call rather than relying on an implicit
    cached login, so this function works the same whether or not the
    environment has ever run `huggingface-cli login`.

    Falls back to `datasets.load_dataset` only if `huggingface_hub` isn't
    available.
    """
    if not token:
        raise ValueError(
            "A Hugging Face access token is required to load Prompt B "
            "datasets (they are gated/private, unlike Prompt A's)."
        )

    try:
        from huggingface_hub import HfApi, hf_hub_download
        import pyarrow.parquet as pq
    except ImportError:
        if load_dataset is None:
            raise ImportError(
                "Neither `huggingface_hub`+`pyarrow` nor `datasets` is "
                "available. Install one of them, or load your data "
                "manually into a DataFrame and pass it straight into "
                "analyze_dataset()."
            )
        ds = load_dataset(repo_id, split=split, token=token)
        return ds.to_pandas()

    api = HfApi(token=token)
    all_files = api.list_repo_files(repo_id, repo_type="dataset")
    parquet_files = [f for f in all_files if f.endswith(".parquet")]


    split_files = [f for f in parquet_files if split in f] or parquet_files
    if not split_files:
        raise FileNotFoundError(
            f"No parquet files found in dataset repo '{repo_id}'. "
            f"Files present: {all_files}"
        )

    frames = []
    for f in split_files:
        local_path = hf_hub_download(repo_id, f, repo_type="dataset", token=token)
        table = pq.read_table(local_path)
        table = _null_columns_to_string(table)
        frames.append(table.to_pandas())

    return pd.concat(frames, ignore_index=True, sort=False)


def clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    Basic hygiene before any stats:
      - keep only successfully-parsed judge responses
      - drop exact duplicate (item_id, judge_model) rows, keeping the latest
        by timestamp (pipeline retries can otherwise create dupes)
    """
    df = df.copy()
    if STATUS_COL in df.columns:
        df = df[df[STATUS_COL] == "ok"]

    if "timestamp" in df.columns:
        df = df.sort_values("timestamp")

    df = df.drop_duplicates(subset=[ITEM_COL, JUDGE_COL], keep="last")
    return df


# --------------------------------------------------------------------------
# 4. KRIPPENDORFF'S ALPHA
# --------------------------------------------------------------------------

def build_reliability_matrix(df: pd.DataFrame, metric: str) -> np.ndarray:
    """
    Reshape long-format (item_id, judge_model, metric) data into the
    (n_judges x n_items) matrix krippendorff.alpha expects.
    Missing judge/item combinations become NaN, which krippendorff
    handles natively as "no rating".
    """
    pivot = df.pivot_table(index=JUDGE_COL, columns=ITEM_COL, values=metric, aggfunc="first")
    return pivot.to_numpy(dtype=float)


def compute_alpha(df: pd.DataFrame, metric: str, level: str) -> float:
    matrix = build_reliability_matrix(df, metric)
    return krippendorff.alpha(reliability_data=matrix, level_of_measurement=level)


def compute_all_alphas(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric, level in METRIC_LEVELS.items():
        if metric not in df.columns:
            continue
        alpha = compute_alpha(df, metric, level)
        n_items = df[ITEM_COL].nunique()
        n_judges = df[JUDGE_COL].nunique()
        rows.append(
            {
                "metric": metric,
                "level_of_measurement": level,
                "krippendorff_alpha": alpha,
                "n_items": n_items,
                "n_judges": n_judges,
            }
        )
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# 5. AGGREGATING JUDGE SCORES PER ITEM
# --------------------------------------------------------------------------

def aggregate_scores(df: pd.DataFrame, agg_funcs=("mean", "median")) -> pd.DataFrame:
    """
    Collapse the 3 (or n) judge rows per item_id into a single row per item,
    with one aggregated column per metric per agg function, plus the raw
    per-judge spread (std) so you can see disagreement at a glance.
    """
    metrics = [m for m in METRIC_LEVELS if m in df.columns]

    agg_map = {m: list(agg_funcs) + ["std", "count"] for m in metrics}
    grouped = df.groupby(ITEM_COL).agg(agg_map)

    grouped.columns = [f"{metric}_{func}" for metric, func in grouped.columns]
    grouped = grouped.reset_index()

    if "system" in df.columns:
        system_map = df.groupby(ITEM_COL)["system"].first()
        grouped = grouped.merge(system_map, on=ITEM_COL, how="left")

    return grouped


# --------------------------------------------------------------------------
# 6. PUTTING IT TOGETHER FOR ONE DATASET
# --------------------------------------------------------------------------

def analyze_dataset(name: str, df: pd.DataFrame):
    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")

    df = clean(df)

    counts = df.groupby(ITEM_COL)[JUDGE_COL].nunique()
    incomplete = counts[counts < counts.max()]
    if len(incomplete):
        print(f"[warning] {len(incomplete)} item(s) have fewer than "
              f"{counts.max()} judge ratings (dropped rows / failed judging).")

    alpha_report = compute_all_alphas(df)
    print("\nKrippendorff's alpha per metric:")
    print(alpha_report.to_string(index=False))

    agg_df = aggregate_scores(df)
    print(f"\nAggregated per-item scores (first 5 of {len(agg_df)}):")
    print(agg_df.head().to_string(index=False))

    return alpha_report, agg_df

In [3]:
def get_output_dir() -> str:
    """
    Resolve a writable directory for outputs.

    Kaggle notebooks can only write to /kaggle/working/ -- anywhere else
    is typically read-only or gets wiped between sessions. This picks
    /kaggle/working if it exists (i.e. we're running on Kaggle),
    otherwise falls back to the current directory.
    """
    kaggle_dir = "/kaggle/working"
    if os.path.isdir(kaggle_dir):
        return kaggle_dir
    return "."


def save_and_link(df: pd.DataFrame, filename: str) -> str:
    """
    Save a DataFrame to CSV in the resolved output directory and, if
    running inside a Jupyter/IPython session (as Kaggle notebooks do),
    print a clickable download link right below the cell -- no need to
    commit the notebook or dig through the Output tab.
    """
    out_dir = get_output_dir()
    path = os.path.join(out_dir, filename)
    df.to_csv(path, index=False)

    try:
        from IPython.display import FileLink, display
        display(FileLink(path))
    except ImportError:
        pass  

    print(f"Saved: {path}")
    return path

In [4]:
def push_results_to_hub(
    name: str,
    alpha_report: pd.DataFrame,
    agg_df: pd.DataFrame,
    token: str,
    private: bool = True,
) -> None:
    """
    Push the Krippendorff alpha report and the per-item aggregated scores
    for one dataset ("exp10", "gpt4.1", ...) to the Hub as a dataset repo.

    Both tables are pushed into the *same* repo (HF_OUTPUT_REPOS[name]) as
    two separate configs, "alpha" and "aggregated", each with a single
    "train" split. That keeps the alpha report and the per-item scores
    together without forcing them into one (mismatched-shape) table.

    Requires a token with write access to the target repo/namespace --
    the same read-scoped token used for loading input data may not be
    enough; use a token with "write" permission, or a fine-grained token
    scoped to the output repo.
    """
    from datasets import Dataset

    repo_id = HF_OUTPUT_REPOS.get(name)
    if not repo_id:
        print(f"[skip] No HF_OUTPUT_REPOS entry for '{name}'; not pushing.")
        return

    Dataset.from_pandas(alpha_report, preserve_index=False).push_to_hub(
        repo_id,
        config_name="alpha",
        split="train",
        token=token,
        private=private,
    )
    Dataset.from_pandas(agg_df, preserve_index=False).push_to_hub(
        repo_id,
        config_name="aggregated",
        split="train",
        token=token,
        private=private,
    )
    print(f"Pushed '{name}' results to https://huggingface.co/datasets/{repo_id} "
          f"(configs: alpha, aggregated)")

In [ ]:
if __name__ == "__main__":
    hf_token = get_hf_token()

    results = {}
    for name, repo_id in HF_DATASETS.items():
        raw_df = load_judged_dataset(repo_id, token=hf_token)
        alpha_report, agg_df = analyze_dataset(name, raw_df)
        results[name] = {"alpha": alpha_report, "aggregated": agg_df}

        save_and_link(alpha_report, f"{name}_promptB_krippendorff_alpha.csv")
        save_and_link(agg_df, f"{name}_promptB_aggregated_scores.csv")

        if PUSH_TO_HUB:
            push_results_to_hub(name, alpha_report, agg_df, token=hf_token, private=PUSH_PRIVATE)

    print(f"\nDone. Files are in: {get_output_dir()}")